In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d

In [ ]:
# Read CSV
df = pd.read_csv('ogd-pollen_pbe_d_recent.csv', sep=';', skiprows=1, encoding='latin-1')

# Convert timestamp column
df['reference_timestamp'] = pd.to_datetime(df['reference_timestamp'], format='%d.%m.%Y %H:%M')

# Keep only data from January to October
df = df[df['reference_timestamp'].dt.month <= 10]

# Select relevant pollen columns (C–I)
pollen_columns = df.columns[2:9]

# Color palette (RGBA for transparency)
colors = [
    'rgba(214, 191, 134, 0.75)',
    'rgba(106, 150, 186, 0.75)',
    'rgba(198, 115, 92, 0.75)',
    'rgba(86, 118, 142, 0.75)',
    'rgba(168, 134, 164, 0.75)',
    'rgba(150, 96, 66, 0.75)',
    'rgba(120, 144, 176, 0.75)'
]

# Convert values to numeric
pollen_data = df[pollen_columns].apply(pd.to_numeric, errors='coerce').fillna(0)

# Create smoother x-axis
x_original = np.arange(len(df))
x_smooth = np.linspace(0, len(df) - 1, len(df) * 5)

# Interpolate timestamps
dates_smooth = pd.date_range(
    start=df['reference_timestamp'].iloc[0],
    end=df['reference_timestamp'].iloc[-1],
    periods=len(x_smooth)
)

# Smooth and interpolate pollen data
pollen_data_smooth = pd.DataFrame()
for col in pollen_columns:
    values = gaussian_filter1d(pollen_data[col].values, sigma=3)
    f = interp1d(x_original, values, kind='cubic', fill_value='extrapolate')
    pollen_data_smooth[col] = np.maximum(f(x_smooth), 0)

# Compute totals for stream centering
totals = pollen_data_smooth.sum(axis=1)
baseline = -totals / 2

# Create figure
fig = go.Figure()

# Invisible baseline
fig.add_trace(go.Scatter(
    x=dates_smooth,
    y=baseline,
    mode='lines',
    line=dict(width=0),
    showlegend=False,
    hoverinfo='skip'
))

# Stack pollen layers
cumulative = baseline.copy()
for col, color in zip(pollen_columns, colors):
    values = pollen_data_smooth[col].values
    y_upper = cumulative + values

    fig.add_trace(go.Scatter(
        x=dates_smooth,
        y=y_upper,
        mode='lines',
        line=dict(width=1.5, color='rgba(255,255,255,0.4)', shape='spline', smoothing=1.3),
        fillcolor=color,
        fill='tonexty',
        name=col,
        hovertemplate='<b>%{fullData.name}</b><br>%{x|%d %b %Y}<br>Value: %{y:.1f}<extra></extra>'
    ))

    cumulative = y_upper

# Generate robust monthly ticks
tickvals = pd.date_range(
    start=dates_smooth.min(),
    end=dates_smooth.max(),
    freq='MS'
)

# Layout
fig.update_layout(
    showlegend=False,
    xaxis=dict(
        showticklabels=True,
        showgrid=False,
        zeroline=False,
        tickmode='array',
        tickvals=tickvals,
        ticktext=['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN',
                  'JUL', 'AUG', 'SEP', 'OCT'],
        tickfont=dict(size=12),
        range=[dates_smooth[0], dates_smooth.max()]
    ),
    yaxis=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False
    ),
    margin=dict(t=20, b=80, l=40, r=40),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    hovermode='x unified',
    height=600,
    width=1200
)

fig.show()

In [ ]:
# Export 
fig.write_image("pollen_stream_graph.svg")